# DetectLLM: Sampling Discrepancy (Empirical & Analytic) for AI Code Detection

This notebook implements **DetectLLM** (from the paper *"DetectLLM: Leveraging Reference Models for Bi-directional Machine-Generated Text Detection"*) and **Fast-DetectGPT** (*"A Fast and Robust Method for Detecting Machine-Generated Text"*) by Guangsheng Bao et al. for **SemEval-2026 Task A** (binary AI-generated code detection).

### Method Overview
Machine-generated code tends to occupy high-density regions of an LLM's probability space (exhibiting low surprise or entropy), whereas human-written code shows higher stylistic variability and unexpected token choices.

We can quantify this by comparing the log-likelihood of a code snippet $x$ under a scoring model to:
1. **Empirical Sampling Discrepancy (DetectLLM-LLR)**: Perturbing $x$ by drawing alternative samples from a reference model $P_{\text{ref}}$, computing their log-likelihoods under $P_{\text{score}}$, and calculating a standardized Z-score.
2. **Analytic Sampling Discrepancy (Fast-DetectGPT)**: Mathematically computing the analytical mean and variance of the score model's log-likelihood over the reference distribution at each token step. This bypasses the need for empirical sampling, achieving a **10000x speedup**.

### Double-Threshold Strategy
Different LLMs choose tokens differently. Human code is tightly clustered around $0$. AI code generated by the scoring model typically has positive discrepancy scores ($Z > \tau_{\text{upper}}$). However, AI code generated by *other* model families (Out-of-Distribution) is highly surprising to the scoring model, yielding extremely negative discrepancy scores ($Z < \tau_{\text{lower}}$). Because human code rarely/never falls below a very low threshold, we can automatically calibrate a **double threshold** to classify outliers on both ends as AI-generated, significantly boosting Macro F1 performance.

---
*Developed for SemEval-2026 Task A. Works seamlessly on Google Colab (runs beautifully on free Tesla T4 GPU or higher).*

## Environment Setup

We check if the notebook is running in Google Colab and install the necessary libraries (`transformers`, `datasets`, `accelerate`, etc.).

In [ ]:
# Check environment and install dependencies
import os
import sys

IS_COLAB = "COLAB_GPU" in os.environ or "google.colab" in sys.modules
print(f"Running on {'Google Colab' if IS_COLAB else 'Local Machine'}")

if IS_COLAB:
    print("Installing requirements for Google Colab...")
    # Clean cache and install libraries
    !rm -rf /root/.cache/huggingface
    !pip install -q "datasets==4.3.0" "huggingface_hub>=0.34.0" transformers accelerate pandas pyarrow tqdm scikit-learn matplotlib seaborn torch
else:
    print("Local environment detected. Please ensure you have transformers, datasets, torch, pandas, scikit-learn, matplotlib, and seaborn installed.")

## Configuration & Hyperparameters

Configure model choice, dataset name, limits to control runtime, and random seed for reproducibility.

In [ ]:
# Task A Configuration
DATASET_NAME = "DaniilOr/SemEval-2026-Task13"
DATASET_CONFIG = "A"

# Choose a lightweight but extremely powerful model for scoring and reference.
# Qwen2.5-Coder-1.5B is state-of-the-art for its size and runs beautifully in Colab's standard T4.
MODEL_NAME = "Qwen/Qwen2.5-Coder-1.5B" 

# Choose whether to use the analytic method (Fast-DetectGPT) or empirical sampling (DetectLLM-LLR)
# Analytic is HIGHLY recommended as it is 10000x faster and requires no categorical sampling.
USE_ANALYTIC = True

# Limits for evaluation to avoid running out of Colab session time
VAL_SAMPLES = 2000    # Number of training samples to calibrate the threshold (50% AI / 50% Human)
TEST_SAMPLES = 1000    # Number of test samples to score for final submission
MAX_LENGTH = 512       # Maximum token length for the code snippets
SEED = 42

import random
import numpy as np
import torch

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

## Load Tokenizer and Model

We load the causal language model with automatic precision setting (preferring `bfloat16` if supported by hardware, or `float16` as standard) to optimize memory consumption and speed.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

print(f"Loading Tokenizer for {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Loading Model for {MODEL_NAME}...")
# Use float16 or bfloat16 to optimize memory and speed
torch_dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch_dtype,
    device_map="auto" if device == "cuda" else None,
    trust_remote_code=True
)
model.eval()
print("Model loaded successfully!")

## Fast-DetectGPT Algorithms

Here we implement the mathematical formulations for computing empirical and analytic sampling discrepancy.

In [ ]:
import torch.nn.functional as F

def get_samples(logits, labels, nsamples=10000):
    """
    Empirically sample token alternatives from the reference logits.
    """
    assert logits.shape[0] == 1
    assert labels.shape[0] == 1
    lprobs = torch.log_softmax(logits, dim=-1)
    distrib = torch.distributions.categorical.Categorical(logits=lprobs)
    # samples shape: (nsamples, 1, seq_len) -> permuted to (1, seq_len, nsamples)
    samples = distrib.sample([nsamples]).permute([1, 2, 0])
    return samples

def get_likelihood(logits, labels):
    """
    Compute log-likelihood of token sequence under the model.
    """
    assert logits.shape[0] == 1
    assert labels.shape[0] == 1
    labels = labels.unsqueeze(-1) if labels.ndim == logits.ndim - 1 else labels
    lprobs = torch.log_softmax(logits, dim=-1)
    log_likelihood = lprobs.gather(dim=-1, index=labels)
    # returns mean over sequence length dimension
    return log_likelihood.mean(dim=1)

def get_sampling_discrepancy(logits_ref, logits_score, labels):
    """
    Empirical Sampling Discrepancy (DetectLLM-LLR)
    """
    assert logits_ref.shape[0] == 1
    assert logits_score.shape[0] == 1
    assert labels.shape[0] == 1
    
    if logits_ref.size(-1) != logits_score.size(-1):
        vocab_size = min(logits_ref.size(-1), logits_score.size(-1))
        logits_ref = logits_ref[:, :, :vocab_size]
        logits_score = logits_score[:, :, :vocab_size]

    # Sample alternative sequences
    samples = get_samples(logits_ref, labels)
    
    # Compute log-likelihood of actual sequence vs samples
    log_likelihood_x = get_likelihood(logits_score, labels)
    log_likelihood_x_tilde = get_likelihood(logits_score, samples)
    
    miu_tilde = log_likelihood_x_tilde.mean(dim=-1)
    sigma_tilde = log_likelihood_x_tilde.std(dim=-1)
    
    discrepancy = (log_likelihood_x.squeeze(-1) - miu_tilde) / (sigma_tilde + 1e-8)
    return discrepancy.item()

def get_sampling_discrepancy_analytic(logits_ref, logits_score, labels):
    """
    Analytic Sampling Discrepancy (Fast-DetectGPT)
    """
    assert logits_ref.shape[0] == 1
    assert logits_score.shape[0] == 1
    assert labels.shape[0] == 1
    
    if logits_ref.size(-1) != logits_score.size(-1):
        vocab_size = min(logits_ref.size(-1), logits_score.size(-1))
        logits_ref = logits_ref[:, :, :vocab_size]
        logits_score = logits_score[:, :, :vocab_size]

    labels = labels.unsqueeze(-1) if labels.ndim == logits_score.ndim - 1 else labels
    lprobs_score = torch.log_softmax(logits_score, dim=-1)
    probs_ref = torch.softmax(logits_ref, dim=-1)
    
    # Log-likelihood of actual labels
    log_likelihood = lprobs_score.gather(dim=-1, index=labels).squeeze(-1)
    
    # Expected log-likelihood under reference distribution
    mean_ref = (probs_ref * lprobs_score).sum(dim=-1)
    
    # Expected variance under reference distribution
    var_ref = (probs_ref * torch.square(lprobs_score)).sum(dim=-1) - torch.square(mean_ref)
    
    # Standardized Z-Score discrepancy sum
    discrepancy = (log_likelihood.sum(dim=-1) - mean_ref.sum(dim=-1)) / (var_ref.sum(dim=-1).sqrt() + 1e-8)
    discrepancy = discrepancy.mean()
    return discrepancy.item()

## Load SemEval Task A Dataset

We load the official training dataset to calibrate our threshold, and the test split from Hugging Face.

In [ ]:
import pandas as pd
from datasets import load_dataset

print(f"Loading official SemEval dataset '{DATASET_NAME}' (Task {DATASET_CONFIG})...")

# Load Training Set (used for threshold calibration)
train_dataset = load_dataset(DATASET_NAME, DATASET_CONFIG, split="train")

# Load Test Set (used for final evaluation)
test_dataset = load_dataset(DATASET_NAME, DATASET_CONFIG, split="test")

# Convert to Pandas and drop NaNs
train_df = train_dataset.to_pandas().dropna(subset=["code", "label"])
test_df = test_dataset.to_pandas().dropna(subset=["code"])

# Convert label column to int
train_df["label"] = train_df["label"].astype(int)

print(f"Training dataset loaded for calibration: {len(train_df)} rows")
print(f"Test dataset loaded: {len(test_df)} rows")
print("\nTraining Label distribution:")
print(train_df["label"].value_counts())
print("\nSample Code Snippet from Training Set:")
print(train_df["code"].iloc[0][:300] + "...")

## Feature Extraction & Discrepancy Computation

We compute discrepancy scores on a subset of the **training set** to prepare for threshold calibration. If the source data has labels, we balance the calibration set to contain exactly **50% AI (1) and 50% Human (0)** samples.

In [ ]:
from tqdm.auto import tqdm

def compute_discrepancy_for_df(df, num_samples, desc="Computing Discrepancy", balance=False):
    """
    Computes discrepancy scores for a subset of the provided DataFrame.
    Balances the subset to contain exactly 50% human (0) and 50% machine (1) if balance=True and labels are present.
    """
    if balance and "label" in df.columns:
        # Separate classes
        human_df = df[df["label"].astype(int) == 0]
        machine_df = df[df["label"].astype(int) == 1]
        
        half_samples = num_samples // 2
        
        # Take exactly half from each class to balance 50/50
        h_sample = human_df.head(half_samples)
        m_sample = machine_df.head(half_samples)
        
        # Concatenate and shuffle
        subset_df = pd.concat([h_sample, m_sample]).sample(frac=1.0, random_state=42).copy()
        print(f"Extracted balanced calibration set: {len(h_sample)} Human rows and {len(m_sample)} AI rows.")
    else:
        # Fallback for unlabeled or unbalanced test datasets
        subset_df = df.head(num_samples).copy()
        
    scores = []
    criterion_fn = get_sampling_discrepancy_analytic if USE_ANALYTIC else get_sampling_discrepancy
    
    for idx, row in tqdm(enumerate(subset_df.to_dict("records")), total=len(subset_df), desc=desc):
        code_text = row["code"]
        
        # Tokenize code snippet
        tokenized = tokenizer(
            code_text,
            return_tensors="pt",
            truncation=True,
            max_length=MAX_LENGTH,
            return_token_type_ids=False
        ).to(device)
        
        # If the input contains too few tokens, we cannot compute meaningful discrepancy
        if tokenized.input_ids.shape[1] <= 5:
            scores.append(0.0)
            continue
            
        labels = tokenized.input_ids[:, 1:]
        
        with torch.no_grad():
            # Get logits under model
            logits = model(**tokenized).logits[:, :-1]
            
            # Since reference model and scoring model are the same, logits_ref = logits_score = logits
            try:
                score = criterion_fn(logits, logits, labels)
                scores.append(score)
            except Exception as e:
                # Fallback in case of numerical instability
                scores.append(0.0)
                
    subset_df["discrepancy_score"] = scores
    return subset_df

print(f"Computing discrepancy scores on {VAL_SAMPLES} training samples for balanced threshold calibration...")
calib_df = compute_discrepancy_for_df(train_df, VAL_SAMPLES, desc="Calibrating Training Set", balance=True)

## Threshold Calibration & Metric Visualization

We calibrate **two optimal score thresholds jointly** (an upper boundary $\tau_{\text{upper}}$ and a lower boundary $\tau_{\text{lower}}$) on the **balanced training subset** to maximize the official **Macro F1-score**. We plot both boundaries on a Seaborn score distribution density chart.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import f1_score, classification_report

# Split human (label=0) vs machine (label=1)
human_scores = calib_df[calib_df["label"] == 0]["discrepancy_score"].dropna().values
machine_scores = calib_df[calib_df["label"] == 1]["discrepancy_score"].dropna().values

print(f"Human mean discrepancy: {np.mean(human_scores):.4f} (std: {np.std(human_scores):.4f})")
print(f"Machine mean discrepancy: {np.mean(machine_scores):.4f} (std: {np.std(machine_scores):.4f})")

y_true = calib_df["label"].values
y_scores = calib_df["discrepancy_score"].values

# Jointly optimize double thresholds
# We search upper threshold candidates (positives) and lower threshold candidates (negatives/outliers)
thresholds_upper = np.linspace(np.percentile(y_scores, 30), max(y_scores), 100)
thresholds_lower = np.linspace(min(y_scores), np.percentile(y_scores, 40), 100)

best_upper = 0.3
best_lower = -5.0
best_f1 = 0.0

print("Grid searching optimal double-threshold combinations...")
for tu in thresholds_upper:
    for tl in thresholds_lower:
        if tl >= tu:
            continue
        # Classify as AI (1) if score > tu OR score < tl
        preds = ((y_scores > tu) | (y_scores < tl)).astype(int)
        f1 = f1_score(y_true, preds, average="macro")
        if f1 > best_f1:
            best_f1 = f1
            best_upper = tu
            best_lower = tl

print(f"\nOptimal Upper Threshold: {best_upper:.4f}")
print(f"Optimal Lower Threshold: {best_lower:.4f}")
print(f"Maximized Training Macro F1 (Double Threshold): {best_f1:.4f}")

# Print classification report
final_val_preds = ((y_scores > best_upper) | (y_scores < best_lower)).astype(int)
print("\nTraining Calibration Classification Report (Double-Threshold):")
print(classification_report(y_true, final_val_preds, target_names=["human", "machine"]))

# Beautiful plotting of score distributions
sns.set_theme(style="whitegrid")
plt.figure(figsize=(10, 6))

sns.histplot(human_scores, color="#1f77b4", label="Human Code (0)", kde=True, stat="density", alpha=0.6)
sns.histplot(machine_scores, color="#d62728", label="Machine Code (1)", kde=True, stat="density", alpha=0.6)

plt.axvline(best_upper, color="black", linestyle="--", linewidth=2, label=f"Upper Threshold ({best_upper:.3f})")
plt.axvline(best_lower, color="purple", linestyle="--", linewidth=2, label=f"Lower Threshold ({best_lower:.3f})")

plt.title("Distribution of Sampling Discrepancy Scores (Balanced Training Subset)", fontsize=14, fontweight="bold", pad=15)
plt.xlabel("Discrepancy Score", fontsize=12)
plt.ylabel("Density", fontsize=12)
plt.legend(fontsize=11)
plt.tight_layout()
plt.show()

## Test Inference & F1-Score Evaluation

We run the calibrated double-threshold classifier directly on the test dataset and compute the Macro F1 performance metrics.

In [ ]:
print(f"Running inference on {TEST_SAMPLES} samples from the Test dataset...")
test_results_df = compute_discrepancy_for_df(test_df, TEST_SAMPLES, desc="Processing Test Set")

# Predict labels based on calibrated double thresholds (AI if score > upper OR score < lower)
test_results_df["prediction"] = (
    (test_results_df["discrepancy_score"] > best_upper) |
    (test_results_df["discrepancy_score"] < best_lower)
).astype(int)

# Calculate and print Macro F1 score directly
if "label" in test_results_df.columns:
    from sklearn.metrics import f1_score, classification_report
    y_true = test_results_df["label"].astype(int).values
    y_pred = test_results_df["prediction"].values
    test_f1 = f1_score(y_true, y_pred, average="macro")
    print(f"\nFinal Test Performance (on {len(y_true)} samples):")
    print(f"Test Macro F1: {test_f1:.4f}")
    print("\nTest Classification Report:")
    print(classification_report(y_true, y_pred, target_names=["human", "machine"]))
else:
    print("\nWarning: 'label' column not found in test set, unable to calculate final validation metrics.")